# My notebook

Run the cell below (the ▶ button) to open Spec Deck here. After a restart it reads your deck from the **My spec deck** cell, which Gemini keeps up to date when you use **Copy for AI**.

In [ ]:
#@title Spec Deck (run this cell to open your deck)
%pip install -q anywidget
import anywidget, traitlets

SPEC_DECK_URL = "https://poolish.org/specdeck-test/?nb=1"   # where Spec Deck lives

class SpecDeck(anywidget.AnyWidget):
    _esm = """
    export default {
      render({ model, el }) {
        const f = document.createElement('iframe');
        f.src = model.get('url');
        f.style.cssText = 'width:100%;height:' + model.get('height') + 'px;border:1px solid #ccc;border-radius:6px';
        f.setAttribute('allow', 'clipboard-read; clipboard-write');
        el.appendChild(f);
        const onMsg = (e) => {
          if (e.source !== f.contentWindow || !e.data) return;
          if (e.data.specdeck === 'ready') f.contentWindow.postMessage({ specdeck: 'load', text: model.get('text') || '' }, '*');
          else if (e.data.specdeck === 'decks' && model.get('text') !== e.data.text) { model.set('text', e.data.text); model.save_changes(); }
        };
        window.addEventListener('message', onMsg);
        return () => window.removeEventListener('message', onMsg);
      }
    };
    """
    url = traitlets.Unicode().tag(sync=True)
    text = traitlets.Unicode("").tag(sync=True)
    height = traitlets.Int(650).tag(sync=True)

def has_boxes(text):
    return any(line.lstrip().startswith('- ') for line in text.splitlines())

def deck_from_cell():
    # Colab only: look through the text cells for the "My spec deck" block
    from google.colab import _message
    notebook = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
    found = ''
    for cell in notebook['cells']:
        if cell['cell_type'] == 'markdown':
            parts = ''.join(cell['source']).split('```specdeck', 1)
            if len(parts) == 2 and '\n```' in parts[1]:
                found = parts[1].split('\n', 1)[1].split('\n```')[0]
    return found

if 'sd' in globals() and has_boxes(sd.text):
    text, source = sd.text, 'kept from this session'
else:
    try:
        text = deck_from_cell()
        source = 'read from your "My spec deck" cell' if text else 'no "My spec deck" cell yet, starting blank'
    except Exception as problem:
        text, source = '', 'could not read the notebook (' + type(problem).__name__ + '), starting blank'

print('Deck:', source)
sd = SpecDeck(url=SPEC_DECK_URL, text=text)
sd